In [3]:
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn

path = "/project/Wellcome_Discovery/datashare/lturiano/data/"

In [4]:
df = pd.read_csv("genes_full_list.txt", sep=",")
df.head()

,Gene stable ID,Gene type,Gene name
0,ENSG00000210049,Mt_tRNA,MT-TF
1,ENSG00000211459,Mt_rRNA,MT-RNR1
2,ENSG00000210077,Mt_tRNA,MT-TV
3,ENSG00000210082,Mt_rRNA,MT-RNR2
4,ENSG00000209082,Mt_tRNA,MT-TL1


In [5]:
df.shape

(86369, 3)

In [6]:
# Make sure there are no duplicates
df = df.drop_duplicates(subset=["Gene stable ID"], keep="first").reset_index(drop=True)

df.shape

(86369, 3)

In [7]:
import pandas as pd

# df columns assumed: "Gene stable ID", "Gene type", "Gene name"

# assign an integer index for embeddings
df["gene_idx"] = range(len(df))

# lookup dicts
ensg_to_idx = dict(zip(df["Gene stable ID"], df["gene_idx"]))
idx_to_ensg = dict(zip(df["gene_idx"], df["Gene stable ID"]))

# optional: symbol lookup (gene symbols are NOT guaranteed unique)
# if you want symbol->idx, decide a rule (e.g., first match)
symbol_to_idx = (
    df.drop_duplicates(subset=["Gene name"], keep="first")
      .set_index("Gene name")["gene_idx"]
      .to_dict()
)

df.head()

,Gene stable ID,Gene type,Gene name,gene_idx
0,ENSG00000210049,Mt_tRNA,MT-TF,0
1,ENSG00000211459,Mt_rRNA,MT-RNR1,1
2,ENSG00000210077,Mt_tRNA,MT-TV,2
3,ENSG00000210082,Mt_rRNA,MT-RNR2,3
4,ENSG00000209082,Mt_tRNA,MT-TL1,4


In [8]:
num_genes = len(df)
emb_dim = 2048  # choose your dimension

gene_embedding = nn.Embedding(num_genes, emb_dim)

In [9]:
# example: get embedding for one gene
i = ensg_to_idx["ENSG00000210049"]
vec = gene_embedding(torch.tensor(i))   # shape: (emb_dim,)

i, vec

(0,
 tensor([ 1.3608, -0.6986, -0.2902,  ..., -0.8896, -1.4851,  0.2607],
        grad_fn=<EmbeddingBackward0>))

## Trying to encode a dataset into the first layer of my VAE

In [10]:
heart_sn = sc.read_h5ad(path + "gex/" + "heart_10x_filt.h5ad")

In [11]:
heart_sn.shape

(63423, 31832)

In [12]:
gene_ids = heart_sn.var_names.astype(str).tolist()
missing = [g for g in gene_ids if g not in ensg_to_idx]
len(missing), missing[:10]

(4,
 ['ENSG00000250410', 'ENSG00000280987', 'ENSG00000266957', 'ENSG00000267058'])

In [14]:
import numpy as np
import scipy.sparse as sp
import anndata as ad
import pandas as pd

def align_adata_to_universe_preserve(
    adata: ad.AnnData,
    universe: list[str],
    var_key: str | None = None,   # None => use adata.var_names; else use adata.var[var_key]
    dtype=np.float32,
    keep_layers: bool = False,    # if True, align each layer too (can be expensive)
):
    # --- gene ids in input (columns of X) ---
    gene_ids = (adata.var_names if var_key is None else adata.var[var_key]).astype(str).to_numpy()

    col_map = {g: i for i, g in enumerate(gene_ids)}
    idx = np.array([col_map.get(g, -1) for g in universe], dtype=np.int64)

    X = adata.X
    n = adata.n_obs
    G = len(universe)

    # --- align X ---
    if sp.issparse(X):
        X = X.tocsr().astype(dtype)
        cols = []
        for src in idx:
            if src == -1:
                cols.append(sp.csr_matrix((n, 1), dtype=dtype))
            else:
                cols.append(X[:, src])
        X_aligned = sp.hstack(cols, format="csr")
    else:
        X = np.asarray(X, dtype=dtype)
        X_aligned = np.zeros((n, G), dtype=dtype)
        keep = idx != -1
        X_aligned[:, keep] = X[:, idx[keep]]

    # --- build var metadata aligned to universe ---
    # We'll use a stable index = the gene IDs used for matching.
    # If var_key is None, adata.var_names are already gene IDs.
    var_in = adata.var.copy()
    if var_key is None:
        var_in = var_in.copy()
        var_in.index = adata.var_names.astype(str)
    else:
        # if var_key column exists, use it as the index for reindexing
        var_in = var_in.copy()
        var_in.index = adata.var[var_key].astype(str).to_numpy()

    var_aligned = var_in.reindex(universe)  # rows for missing genes become NaN
    var_aligned.index = pd.Index(universe, name=var_in.index.name)

    # --- create output AnnData and preserve metadata ---
    out = ad.AnnData(
        X=X_aligned,
        obs=adata.obs.copy(),
        var=var_aligned,
        uns=adata.uns.copy(),
        obsm=adata.obsm.copy(),
        varm=adata.varm.copy(),
        obsp=adata.obsp.copy(),
        varp=adata.varp.copy(),
    )
    out.var_names = universe

    # --- optionally align layers too ---
    if keep_layers and len(adata.layers) > 0:
        for lname, L in adata.layers.items():
            if sp.issparse(L):
                L = L.tocsr().astype(dtype)
                cols = []
                for src in idx:
                    if src == -1:
                        cols.append(sp.csr_matrix((n, 1), dtype=dtype))
                    else:
                        cols.append(L[:, src])
                out.layers[lname] = sp.hstack(cols, format="csr")
            else:
                L = np.asarray(L, dtype=dtype)
                L_al = np.zeros((n, G), dtype=dtype)
                keep = idx != -1
                L_al[:, keep] = L[:, idx[keep]]
                out.layers[lname] = L_al

    return out, idx

In [15]:
# fixed gene universe in canonical order (length = ~80k)
universe = df.sort_values("gene_idx")["Gene stable ID"].astype(str).tolist()

# sanity checks
assert len(universe) == len(set(universe)), "ENSG universe has duplicates"
assert universe[0] == idx_to_ensg[0]
assert universe[-1] == idx_to_ensg[len(universe)-1]

In [16]:
heart_sn_aligned, idx = align_adata_to_universe_preserve(heart_sn, universe, var_key=None)
print(heart_sn_aligned.shape)  # (45781, 86369)

(63423, 86369)


In [17]:
heart_sn_aligned.var_names

Index(['ENSG00000210049', 'ENSG00000211459', 'ENSG00000210077',
       'ENSG00000210082', 'ENSG00000209082', 'ENSG00000198888',
       'ENSG00000210100', 'ENSG00000210107', 'ENSG00000210112',
       'ENSG00000198763',
       ...
       'ENSG00000143515', 'ENSG00000189171', 'ENSG00000152382',
       'ENSG00000143740', 'ENSG00000117472', 'ENSG00000168710',
       'ENSG00000081692', 'ENSG00000157873', 'ENSG00000132676',
       'ENSG00000163374'],
      dtype='object', length=86369)

In [18]:
universe

['ENSG00000210049',
 'ENSG00000211459',
 'ENSG00000210077',
 'ENSG00000210082',
 'ENSG00000209082',
 'ENSG00000198888',
 'ENSG00000210100',
 'ENSG00000210107',
 'ENSG00000210112',
 'ENSG00000198763',
 'ENSG00000210117',
 'ENSG00000210127',
 'ENSG00000210135',
 'ENSG00000210140',
 'ENSG00000210144',
 'ENSG00000198804',
 'ENSG00000210151',
 'ENSG00000210154',
 'ENSG00000198712',
 'ENSG00000210156',
 'ENSG00000228253',
 'ENSG00000198899',
 'ENSG00000198938',
 'ENSG00000210164',
 'ENSG00000198840',
 'ENSG00000210174',
 'ENSG00000212907',
 'ENSG00000198886',
 'ENSG00000210176',
 'ENSG00000210184',
 'ENSG00000210191',
 'ENSG00000198786',
 'ENSG00000198695',
 'ENSG00000210194',
 'ENSG00000198727',
 'ENSG00000210195',
 'ENSG00000210196',
 'ENSG00000299200',
 'ENSG00000308964',
 'ENSG00000278457',
 'ENSG00000271254',
 'ENSG00000278704',
 'ENSG00000278625',
 'ENSG00000309338',
 'ENSG00000295063',
 'ENSG00000309357',
 'ENSG00000309229',
 'ENSG00000310277',
 'ENSG00000304745',
 'ENSG00000276197',


## Try to align a dataset to the gene universe

In [2]:
heart_sc = sc.read_h5ad(path + "rna/" + "heart_rna_filt.h5ad")

NameError: name 'sc' is not defined

In [1]:
heart_sc

NameError: name 'heart_sc' is not defined

In [65]:
heart_sc_aligned.var

,feature_name
ENSG00000210049,NaN
ENSG00000211459,NaN
ENSG00000210077,NaN
ENSG00000210082,NaN
ENSG00000209082,NaN
...,...
ENSG00000168710,AHCYL1
ENSG00000081692,JMJD4
ENSG00000157873,TNFRSF14
ENSG00000132676,DAP3


## Protein-coding genes only

In [7]:
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn

path = "/project/Wellcome_Discovery/datashare/lturiano/data/"

In [8]:
df = pd.read_csv("protein_coding_genes_list.txt", sep=",")
df.head()

,Gene stable ID,Gene type,Gene name
0,ENSG00000198888,protein_coding,MT-ND1
1,ENSG00000198763,protein_coding,MT-ND2
2,ENSG00000198804,protein_coding,MT-CO1
3,ENSG00000198712,protein_coding,MT-CO2
4,ENSG00000228253,protein_coding,MT-ATP8


In [9]:
df.shape

(23262, 3)

In [10]:
import numpy as np
import scipy.sparse as sp
import anndata as ad
import pandas as pd

def align_adata_to_universe_preserve(
    adata: ad.AnnData,
    universe: list[str],
    var_key: str | None = None,   # None => use adata.var_names; else use adata.var[var_key]
    dtype=np.float32,
    keep_layers: bool = False,    # if True, align each layer too (can be expensive)
):
    # --- gene ids in input (columns of X) ---
    gene_ids = (adata.var_names if var_key is None else adata.var[var_key]).astype(str).to_numpy()

    col_map = {g: i for i, g in enumerate(gene_ids)}
    idx = np.array([col_map.get(g, -1) for g in universe], dtype=np.int64)

    X = adata.X
    n = adata.n_obs
    G = len(universe)

    # --- align X ---
    if sp.issparse(X):
        X = X.tocsr().astype(dtype)
        cols = []
        for src in idx:
            if src == -1:
                cols.append(sp.csr_matrix((n, 1), dtype=dtype))
            else:
                cols.append(X[:, src])
        X_aligned = sp.hstack(cols, format="csr")
    else:
        X = np.asarray(X, dtype=dtype)
        X_aligned = np.zeros((n, G), dtype=dtype)
        keep = idx != -1
        X_aligned[:, keep] = X[:, idx[keep]]

    # --- build var metadata aligned to universe ---
    # We'll use a stable index = the gene IDs used for matching.
    # If var_key is None, adata.var_names are already gene IDs.
    var_in = adata.var.copy()
    if var_key is None:
        var_in = var_in.copy()
        var_in.index = adata.var_names.astype(str)
    else:
        # if var_key column exists, use it as the index for reindexing
        var_in = var_in.copy()
        var_in.index = adata.var[var_key].astype(str).to_numpy()

    var_aligned = var_in.reindex(universe)  # rows for missing genes become NaN
    var_aligned.index = pd.Index(universe, name=var_in.index.name)

    # --- create output AnnData and preserve metadata ---
    out = ad.AnnData(
        X=X_aligned,
        obs=adata.obs.copy(),
        var=var_aligned,
        uns=adata.uns.copy(),
        obsm=adata.obsm.copy(),
        varm=adata.varm.copy(),
        obsp=adata.obsp.copy(),
        varp=adata.varp.copy(),
    )
    out.var_names = universe

    # --- optionally align layers too ---
    if keep_layers and len(adata.layers) > 0:
        for lname, L in adata.layers.items():
            if sp.issparse(L):
                L = L.tocsr().astype(dtype)
                cols = []
                for src in idx:
                    if src == -1:
                        cols.append(sp.csr_matrix((n, 1), dtype=dtype))
                    else:
                        cols.append(L[:, src])
                out.layers[lname] = sp.hstack(cols, format="csr")
            else:
                L = np.asarray(L, dtype=dtype)
                L_al = np.zeros((n, G), dtype=dtype)
                keep = idx != -1
                L_al[:, keep] = L[:, idx[keep]]
                out.layers[lname] = L_al

    return out, idx